# Experimentos — Baselines (Popularity e SVD)

Notebook de experimentação dos modelos baseline antes da formalização em `train.py`.
Todos os experimentos são rastreados no MLflow via `start_notebook_run()`.

## 1. Setup

Imports, `configure_mlflow_tracking()` e `get_settings()`.

## 2. Carregamento e preparação dos dados

`load_dataset()` → `build_interactions()` → `temporal_split()`

Resultado: `train_df`, `val_df`, `test_df` (split temporal 60/20/20).
O `test_df` é tocado apenas uma vez, na avaliação final.

## 3. Loop de avaliação

Função local `evaluate_model(model, split_df, k)` que recebe qualquer `BaseRecommender`
e retorna as 6 métricas padronizadas. Reutilizada por todos os experimentos abaixo.

**Métricas retornadas:** Precision@K, Recall@K, NDCG@K, Hit Rate@K, Coverage, Revenue@K

**Ground truth:** apenas interações do tipo `transaction`.

## 4. Baseline: PopularityRecommender

Modelo sem personalização — recomenda os itens mais populares por soma de score.
Equivalente ao `DummyClassifier`: define o piso de desempenho que qualquer modelo
precisa superar.

Experimento único (sem hiperparâmetros para variar).
`phase="baseline"`, avaliado no `val_df`.

## 5. Baseline: SVDRecommender

Fatoração de matriz via `scikit-surprise`. Personaliza recomendações por usuário,
ao contrário do Popularity.

### Variações testadas

| Run | `n_factors` | `n_epochs` | Objetivo |
|-----|-------------|------------|----------|
| svd-nf25 | 25 | 20 | Referência de underfitting |
| svd-nf50 | 50 | 20 | Default do projeto (`SVD_N_FACTORS`) |
| svd-nf100 | 100 | 20 | Maior capacidade |
| svd-nf50-ep50 | 50 | 50 | Convergência do default |

Cada variação é um run separado no MLflow com o mesmo `run_group`.
Critério de seleção: **NDCG@10** (desempate: Hit Rate@10).

### Por que não cross validation?
Split temporal não pode ser embaralhado — CV clássico vazaria interações futuras
para o treino. As variações são avaliadas sempre no mesmo `val_df`.

## 6. Comparativo final

`MlflowClient` puxa os runs do experimento e monta um DataFrame de comparação
Popularity vs melhor SVD em todas as 6 métricas.

Este resultado justifica o investimento no MLP (`03_mlp.ipynb`):
- Métricas de modelo provam que o SVD aprende padrões reais
- Métricas de negócio (Coverage, Revenue@K) traduzem o ganho em valor